# Misc: E-Commerce Scraping (extruct, price-parser, Scrapy)

Tidak ada satu library "ajaib" untuk e-commerce (seperti `newspaper` untuk berita), tapi
ada beberapa tool yang sangat membantu. Notebook ini membahas:

1. **`extruct`** — ambil **structured data** (JSON-LD `schema.org/Product`) yang sering
   ditanam situs e-commerce. (contoh **berhasil** & **gagal**)
2. **`price-parser`** — rapikan string harga yang formatnya beragam jadi angka.
3. **robots.txt** — apa itu, dan cara scraping yang **baik vs buruk** (patuh vs abai).
4. **`Scrapy`** — framework crawling: **engine async**, kontrol konkurensi (with vs without,
   bandingkan waktu), retry, throttling, dan `ROBOTSTXT_OBEY`.
5. **`scrapy-poet` + `zyte-common-items`** — pola **page object** agar kode rapi & reusable.

Install:

```bash
uv sync --extra ecommerce
```

**Tooling:** `extruct`, `price-parser`, `scrapy` (+ `scrapy-poet`, `zyte-common-items` opsional).


## 1. `extruct` — Structured Data (JSON-LD)

Banyak situs e-commerce menanam data produk dalam format **schema.org** (biasanya JSON-LD)
di dalam `<script type="application/ld+json">`. Tujuannya untuk SEO/Google, tapi sangat
berguna buat kita: **nama, harga, currency, stok, rating** sudah terstruktur rapi —
tinggal ambil, tanpa nulis selector.

`extruct` mengekstrak JSON-LD / Microdata / RDFa / OpenGraph sekaligus.

### Contoh BERHASIL — situs yang menyediakan JSON-LD Product


In [1]:
import extruct

# Halaman yang BAIK: menanam JSON-LD Product (seperti banyak toko online sungguhan)
html_ok = """
<html><head>
<script type="application/ld+json">
{
  "@context": "https://schema.org",
  "@type": "Product",
  "name": "Kopi Arabika Gayo 250g",
  "brand": "TaniKopi",
  "sku": "KOPI-250",
  "offers": {
    "@type": "Offer",
    "price": "85000",
    "priceCurrency": "IDR",
    "availability": "https://schema.org/InStock"
  },
  "aggregateRating": {"@type": "AggregateRating", "ratingValue": "4.7", "reviewCount": "128"}
}
</script>
</head><body><h1>Kopi Arabika Gayo</h1></body></html>
"""

data = extruct.extract(html_ok, syntaxes=["json-ld", "microdata", "opengraph"])
produk = data["json-ld"][0]  # ambil Product pertama

print("Nama     :", produk["name"])
print("Brand    :", produk["brand"])
print("Harga    :", produk["offers"]["price"], produk["offers"]["priceCurrency"])
print("Stok     :", produk["offers"]["availability"])
print("Rating   :", produk["aggregateRating"]["ratingValue"])


Nama     : Kopi Arabika Gayo 250g
Brand    : TaniKopi
Harga    : 85000 IDR
Stok     : https://schema.org/InStock
Rating   : 4.7


### Contoh GAGAL — situs tanpa structured data

Tidak semua situs menanam JSON-LD. Kalau tidak ada, `extruct` mengembalikan list **kosong**.
Di sinilah kita harus **fallback** ke cara manual (BeautifulSoup / XPath). Pola yang baik:
coba `extruct` dulu (cepat & bersih), kalau kosong baru parsing manual.


In [2]:
from bs4 import BeautifulSoup

# Halaman yang BURUK (untuk kita): tidak ada structured data sama sekali
html_bad = """
<html><body>
  <div class="product">
    <span class="title">Kopi Robusta 250g</span>
    <span class="price">Rp65.000</span>
  </div>
</body></html>
"""

hasil = extruct.extract(html_bad, syntaxes=["json-ld", "microdata", "opengraph"])
print("Jumlah data terstruktur ditemukan:", {k: len(v) for k, v in hasil.items()})

# Karena kosong -> fallback ke parsing manual
if not hasil["json-ld"]:
    print("-> Tidak ada JSON-LD. Fallback ke BeautifulSoup.")
    soup = BeautifulSoup(html_bad, "html.parser")
    print("Nama :", soup.find("span", class_="title").get_text(strip=True))
    print("Harga:", soup.find("span", class_="price").get_text(strip=True))


Jumlah data terstruktur ditemukan: {'microdata': 0, 'json-ld': 0, 'opengraph': 0}
-> Tidak ada JSON-LD. Fallback ke BeautifulSoup.
Nama : Kopi Robusta 250g
Harga: Rp65.000


## 2. `price-parser` — Rapikan Harga

Harga di web formatnya **kacau** dan beda-beda per negara/situs: `Rp75.000`,
`$1,234.56`, `€ 1.234,56`, `Rp350.000,-`. Menulis regex sendiri untuk semua kasus itu
melelahkan dan rawan salah (titik = ribuan atau desimal?).

`price-parser` menangani ini otomatis: kembalikan `amount` (angka) + `currency`.


In [3]:
from price_parser import Price

contoh = [
    "Rp 1.250.000",
    "Rp75.000",
    "Harga: Rp350.000,-",
    "$1,234.56",
    "€ 1.234,56",
    "USD 19.99",
]

print(f"{'input':25} {'amount':>12}  currency")
print("-" * 50)
for s in contoh:
    p = Price.fromstring(s)
    print(f"{s:25} {str(p.amount):>12}  {p.currency}")


input                           amount  currency
--------------------------------------------------
Rp 1.250.000                   1250000  Rp
Rp75.000                         75000  Rp
Harga: Rp350.000,-              350000  Rp
$1,234.56                      1234.56  $
€ 1.234,56                     1234.56  €
USD 19.99                        19.99  USD


## 3. robots.txt & Etika Scraping

**`robots.txt`** adalah file di root sebuah situs (mis. `https://contoh.com/robots.txt`)
yang berisi **aturan untuk bot/crawler**: bagian mana yang boleh (`Allow`) dan tidak boleh
(`Disallow`) diakses, dan kadang `Crawl-delay`. Ini **konvensi sopan santun**, bukan
penghalang teknis — tapi mengabaikannya bisa melanggar Terms of Service dan bikin IP diblok.

Python punya pembaca bawaan: `urllib.robotparser`.

In [4]:
from urllib.robotparser import RobotFileParser

rp = RobotFileParser()
rp.set_url("https://www.google.com/robots.txt")
try:
    rp.read()
    uji = [
        "https://www.google.com/search?q=sepatu",  # biasanya Disallow
        "https://www.google.com/",                 # biasanya Allow
    ]
    for url in uji:
        boleh = rp.can_fetch("*", url)
        print(("BOLEH " if boleh else "DILARANG"), "->", url)
except Exception as e:
    print("Gagal baca robots.txt:", e)

DILARANG -> https://www.google.com/search?q=sepatu
DILARANG -> https://www.google.com/


### Scraping yang BAIK vs BURUK

| Aspek | ✅ Baik | ❌ Buruk |
| --- | --- | --- |
| robots.txt | dicek & dipatuhi | diabaikan |
| User-Agent | jelas / jujur | kosong atau menyamar |
| Kecepatan | ada jeda (delay), 1 request/detik wajar | hajar secepat mungkin |
| Konkurensi | dibatasi wajar | ribuan request paralel ke 1 situs |
| Beban server | minimal, ambil seperlunya | bikin situs lambat/down (mirip DDoS) |
| Data | hormati ToS & data pribadi | scrape data sensitif |

Di bawah: `PoliteFetcher` yang **mengecek robots.txt + memberi jeda** sebelum mengambil.

In [5]:
import time

import requests
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser


class PoliteFetcher:
    """Fetcher 'sopan': cek robots.txt + kasih jeda + User-Agent jelas."""

    def __init__(self, user_agent="LatihanScrapingBot/1.0 (+belajar)", delay=1.0):
        self.ua = user_agent
        self.delay = delay
        self._cache = {}

    def _robot(self, url):
        base = "{0.scheme}://{0.netloc}".format(urlparse(url))
        if base not in self._cache:
            rp = RobotFileParser()
            rp.set_url(base + "/robots.txt")
            try:
                rp.read()
            except Exception:
                rp = None  # tidak ada robots.txt -> anggap boleh
            self._cache[base] = rp
        return self._cache[base]

    def get(self, url):
        rp = self._robot(url)
        if rp and not rp.can_fetch(self.ua, url):
            raise PermissionError(f"Dilarang oleh robots.txt: {url}")
        time.sleep(self.delay)  # jeda biar tidak membebani server
        return requests.get(url, headers={"User-Agent": self.ua}, timeout=10)


fetcher = PoliteFetcher(delay=0.5)
resp = fetcher.get("https://books.toscrape.com/")
print("Sukses (sopan):", resp.status_code, "| panjang HTML:", len(resp.text))

# ❌ Cara BURUK (JANGAN dilakukan): tanpa cek robots, tanpa UA, tanpa jeda,
#    dan dengan konkurensi tinggi ke satu situs:
#       for url in ribuan_url:
#           requests.get(url)          # spam -> server kewalahan, IP diblok
print("\n(Contoh buruk hanya dijelaskan, tidak dijalankan.)")

Sukses (sopan): 200 | panjang HTML: 51294

(Contoh buruk hanya dijelaskan, tidak dijalankan.)


## 4. Scrapy — kenapa butuh framework?

Untuk e-commerce **nyata** (ribuan produk, banyak halaman), `requests` + loop biasa terasa
lambat dan repot. `requests.get()` itu **sinkron**: satu per satu, nunggu response selesai
baru lanjut. Kalau tiap request 0,3 detik, 1000 request = 5 menit hanya untuk menunggu.

**Scrapy** dibangun di atas engine **asynchronous** (Twisted/asyncio): banyak request
"berjalan bersamaan" tanpa saling menunggu (cocok untuk pekerjaan I/O seperti jaringan).
Selain itu Scrapy memberi **gratis**:

- **Kontrol konkurensi**: `CONCURRENT_REQUESTS`, `CONCURRENT_REQUESTS_PER_DOMAIN`.
- **Sopan otomatis**: `ROBOTSTXT_OBEY`, `DOWNLOAD_DELAY`, `AUTOTHROTTLE` (atur kecepatan sendiri).
- **Tangguh**: retry otomatis, timeout, caching, dedup URL.
- **Pipeline**: bersihkan & simpan item (mis. pakai `price-parser`) secara terstruktur.

Di bawah kita tulis satu spider untuk `books.toscrape.com` yang mengambil judul + harga
(dibersihkan dengan `price-parser`).

> Catatan: Scrapy memakai reactor Twisted yang tidak bisa di-restart dalam satu proses,
> jadi di notebook kita jalankan spider lewat **subprocess** (`scrapy runspider`) — cara yang
> bersih & sama seperti di produksi.

In [6]:
import tempfile
import textwrap
from pathlib import Path

# Tulis kode spider ke sebuah file (.py) untuk dijalankan dengan `scrapy runspider`.
SPIDER_CODE = textwrap.dedent(
    '''
    import scrapy
    from price_parser import Price

    class BooksSpider(scrapy.Spider):
        name = "books"
        # custom_settings = setelan khusus spider ini
        custom_settings = {
            "ROBOTSTXT_OBEY": True,        # patuhi robots.txt
            "USER_AGENT": "LatihanScrapingBot/1.0 (+belajar)",
            "LOG_LEVEL": "ERROR",          # output bersih
            # untuk produksi yang sopan, aktifkan AutoThrottle:
            # "AUTOTHROTTLE_ENABLED": True,
        }
        # ambil 12 halaman katalog
        start_urls = [
            f"https://books.toscrape.com/catalogue/page-{i}.html"
            for i in range(1, 13)
        ]

        def parse(self, response):
            for art in response.css("article.product_pod"):
                harga_teks = art.css("p.price_color::text").get()
                yield {
                    "judul": art.css("h3 a::attr(title)").get(),
                    # price-parser: "£51.77" -> 51.77 (float)
                    "harga": Price.fromstring(harga_teks).amount_float,
                }
    '''
)

SPIDER_PATH = str(Path(tempfile.gettempdir()) / "books_spider.py")
Path(SPIDER_PATH).write_text(SPIDER_CODE)
print("Spider ditulis ke:", SPIDER_PATH)
print(SPIDER_CODE)

Spider ditulis ke: /var/folders/lv/0w17hlfs4073bf5lwh3_79wr0000gn/T/books_spider.py

import scrapy
from price_parser import Price

class BooksSpider(scrapy.Spider):
    name = "books"
    # custom_settings = setelan khusus spider ini
    custom_settings = {
        "ROBOTSTXT_OBEY": True,        # patuhi robots.txt
        "USER_AGENT": "LatihanScrapingBot/1.0 (+belajar)",
        "LOG_LEVEL": "ERROR",          # output bersih
        # untuk produksi yang sopan, aktifkan AutoThrottle:
        # "AUTOTHROTTLE_ENABLED": True,
    }
    # ambil 12 halaman katalog
    start_urls = [
        f"https://books.toscrape.com/catalogue/page-{i}.html"
        for i in range(1, 13)
    ]

    def parse(self, response):
        for art in response.css("article.product_pod"):
            harga_teks = art.css("p.price_color::text").get()
            yield {
                "judul": art.css("h3 a::attr(title)").get(),
                # price-parser: "£51.77" -> 51.77 (float)
                "harga": P

### Async: dengan vs tanpa konkurensi (bandingkan waktu)

Spider yang sama dijalankan dua kali, hanya beda `CONCURRENT_REQUESTS`:

- `CONCURRENT_REQUESTS=1` → praktis **sinkron** (satu request pada satu waktu).
- `CONCURRENT_REQUESTS=16` → **async**, banyak request berjalan bersamaan.

Jumlah item harus sama, tapi waktunya jauh berbeda. (`DOWNLOAD_DELAY=0` agar yang diukur
murni efek konkurensi; di produksi tetap beri delay demi kesopanan.)

In [7]:
import json
import subprocess
import sys
import time


def jalankan_scrapy(concurrency, out_file):
    t0 = time.perf_counter()
    subprocess.run(
        [
            sys.executable, "-m", "scrapy", "runspider", SPIDER_PATH,
            "-s", f"CONCURRENT_REQUESTS={concurrency}",
            "-s", "DOWNLOAD_DELAY=0",
            "-s", "LOG_LEVEL=ERROR",
            "-O", f"{out_file}:json",  # -O = timpa file
        ],
        check=True,
    )
    durasi = time.perf_counter() - t0
    jumlah = len(json.load(open(out_file)))
    return durasi, jumlah


d1, n1 = jalankan_scrapy(1, "/tmp/scrapy_sinkron.json")
d16, n16 = jalankan_scrapy(16, "/tmp/scrapy_async.json")

print(f"CONCURRENT_REQUESTS=1  (sinkron) : {d1:5.1f} s   ({n1} item)")
print(f"CONCURRENT_REQUESTS=16 (async)   : {d16:5.1f} s   ({n16} item)")
print(f"\n=> Async ~{d1 / d16:.1f}x lebih cepat untuk jumlah data yang sama.")

CONCURRENT_REQUESTS=1  (sinkron) :   6.3 s   (240 item)
CONCURRENT_REQUESTS=16 (async)   :   3.0 s   (240 item)

=> Async ~2.1x lebih cepat untuk jumlah data yang sama.


### Hasil Scrapy → pandas (nyambung ke cleaning & DB)

Output Scrapy (file JSON) tinggal dibaca ke `DataFrame`. Karena harga sudah dibersihkan
`price-parser` saat scraping, kolom `harga` langsung bertipe angka — siap masuk ke tahap
**cleaning / simpan ke database** seperti di notebook `walkthrough.ipynb`.

In [8]:
import pandas as pd

# baca hasil crawl Scrapy (dari run async di atas)
df = pd.read_json("/tmp/scrapy_async.json")

print("Bentuk DataFrame:", df.shape)
print("\nTipe data:")
print(df.dtypes)

print("\nStatistik harga:")
print(f"  termurah  : {df['harga'].min():.2f}")
print(f"  termahal  : {df['harga'].max():.2f}")
print(f"  rata-rata : {df['harga'].mean():.2f}")

df.head()

Bentuk DataFrame: (240, 2)

Tipe data:
judul        str
harga    float64
dtype: object

Statistik harga:
  termurah  : 10.16
  termahal  : 59.64
  rata-rata : 34.62


,judul,harga
0,A Light in the Attic,51.77
1,Tipping the Velvet,53.74
2,Soumission,50.10
3,Sharp Objects,47.82
4,Sapiens: A Brief History of Humankind,54.23


## 5. `scrapy-poet` + `zyte-common-items` — kode rapi & reusable

Saat proyek membesar, logika ekstraksi (selector) bercampur di dalam spider jadi susah
dirawat. **`scrapy-poet`** memperkenalkan pola **Page Object**: pisahkan "cara mengekstrak
satu halaman" ke kelas tersendiri. Spider cukup minta hasilnya. Keuntungannya:

- **Terpisah & rapi**: selector ada di Page Object, bukan berserakan di spider.
- **Reusable & testable**: Page Object bisa dipakai ulang dan diuji tanpa menjalankan crawl.
- **Standar item**: **`zyte-common-items`** menyediakan skema siap pakai seperti `Product`
  (name, price, currency, availability, sku, images, ...) sehingga output konsisten.
- **Bisa dipadukan dengan ekstraksi AI Zyte** (otomatis mengisi field Product) untuk situs
  yang tidak punya structured data.

Contoh pola (perlu setup project Scrapy + konfigurasi `scrapy-poet`, jadi **tidak dijalankan
di notebook**):

```python
import attrs
from web_poet import WebPage, field
from zyte_common_items import Product
import scrapy

# Page Object: semua logika ekstraksi 1 halaman produk ada di sini
@attrs.define
class BookProductPage(WebPage):
    @field
    def name(self) -> str:
        return self.css("h1::text").get()

    @field
    def price(self) -> str:
        return self.css("p.price_color::text").get()

    def to_item(self) -> Product:
        return Product(name=self.name, price=self.price, currencyRaw="GBP")

# Spider tinggal MINTA Product, tidak tahu-menahu soal selector
class BooksSpider(scrapy.Spider):
    name = "books_poet"
    start_urls = ["https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"]

    def parse(self, response, page: BookProductPage):  # page disuntik scrapy-poet
        yield page.to_item()
```

Install (opsional): `uv sync --extra ecommerce-advanced`.

## Kesimpulan — kapan pakai apa (e-commerce)

1. **Cek structured data dulu** (`extruct` → JSON-LD `Product`). Kalau ada, paling bersih.
2. **Tidak ada?** Fallback ke BeautifulSoup / XPath / hidden API.
3. **Harga berantakan?** Rapikan dengan `price-parser`.
4. **Skala besar / banyak halaman?** Pakai **Scrapy** (async, retry, throttle, robots).
5. **Proyek serius & jangka panjang?** Rapikan dengan **`scrapy-poet`** + **`zyte-common-items`**.
6. **Selalu** patuhi `robots.txt`, beri jeda, pakai User-Agent jelas, dan jangan membebani server.

> Ringkas: *mulai dari yang paling ringan & sopan, naik kelas hanya saat benar-benar perlu.*